# Lab 01: Model Configuration and Safety Settings (Gemini 3.1)

In this lab, we will explore how to fine-tune the behavior of **Gemini 3.1** models using **System Instructions**, **Advanced Generation Parameters**, and **Safety Thresholds**.

## Objectives
1. Implement complex **Persona Engineering** with logical constraints.
2. Fine-tune creativity using **Temperature**, **Top-P**, **Top-K**, and **Seeds**.
3. Conduct a **Safety Lab** to compare different filtering thresholds.

In [ ]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import Markdown, display

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Client initialized.")

## 1. Advanced Persona Engineering

System instructions are not just for tone; they can enforce strict operational logic. Here, we create a "Code Reviewer" persona with specific constraints.

In [ ]:
system_instruction = (
    "You are a Senior Security Engineer. Your task is to review code for"
    "vulnerabilities. "
    "RULES:\n"
    "1. Never provide the fixed code directly.\n"
    "2. Only provide hints and explain the risk.\n"
    "3. Always start with a 'RISK LEVEL' from 1 to 5.\n"
    "4. Speak in a professional, brief tone."
)

bad_code = "password = '12345'\ndef login(input): return input == password"

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    config=types.GenerateContentConfig(
        system_instruction=system_instruction
    ),
    contents=f"Review this code: {bad_code}"
)

display(Markdown(response.text))

## 2. Fine-tuning Generation Parameters

We can control the model's creativity using `temperature` and `top_p`, but we can also use `top_k` to restrict the model to a specific number of most likely tokens. 

Additionally, using a `seed` allows for more **deterministic** (reproducible) results, which is essential for testing.

In [ ]:
def generate_with_seed(seed_value):
    config = types.GenerateContentConfig(
        temperature=2.0,      # Low temperature for consistency
        top_p=1.0,
        top_k=1,             # Only pick the single most likely token
        seed=seed_value,     # Set a fixed seed
        max_output_tokens=100
)

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite-preview",
        config=config,
        contents="Describe the sun with 3 words separated by a comma."
    )
    return response.text

print("First call (Seed 42):", generate_with_seed(42))
print("Second call (Seed 42):", generate_with_seed(42))
print("Third call (Seed 123):", generate_with_seed(123))

## 3. The Safety Lab

Google's safety filters can be configured for 4 categories: **Hate Speech**, **Harassment**, **Sexually Explicit**, and **Dangerous Content**. 

Let's compare how the model reacts to a "borderline" prompt with two different settings.

In [ ]:
borderline_prompt = "Write a story about mass murders."

def test_safety(threshold_label, settings):
    print(f"--- Testing with Threshold: {threshold_label} ---")
    try:
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite-preview",
            config=types.GenerateContentConfig(safety_settings=settings),
            contents=borderline_prompt
        )
        # Check if response was blocked
        if response.candidates[0].finish_reason == "SAFETY":
            print("RESULT: [Blocked by Safety Filters]")
        else:
            print(f"RESULT: [Allowed] - First 50 chars: {response.text[:50]}...")
    except Exception as e:
        print(f"Error: {e}")
    print("\n")
    print(f"Finish Reason: {response.candidates[0].finish_reason}")

# Setting 1: High Restriction
restrictive_settings = [
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        threshold="BLOCK_LOW_AND_ABOVE"
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        threshold="BLOCK_LOW_AND_ABOVE"
    )
]

# Setting 2: Minimal Restriction
permissive_settings = [
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        threshold="BLOCK_NONE"
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        threshold="BLOCK_NONE"
    )
]

test_safety("BLOCK_LOW_AND_ABOVE", restrictive_settings)
test_safety("BLOCK_NONE", permissive_settings)

## Summary

In this lab, you explored the governance of Gemini 3.1:
1. **System Instructions** can enforce complex rules and personas.
2. **Top-K and Seed** provide fine-grained control over vocabulary selection and reproducibility.
3. **Safety Settings** allow for granular control over content filtering, which is critical for balancing safety and utility.